In [21]:
import yfinance as yf
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

In [5]:
goog = yf.download("GOOG", start = "2008-01-01", end = "2018-01-01")

[*********************100%***********************]  1 of 1 completed


In [12]:
x = pd.DataFrame(index = goog.index)
x['daily returns'] = goog['Close'].pct_change()

In [40]:
win1 = 10
win2 = 20
roll_mean_list_10 = []
roll_std_list_10 = []
roll_mean_list_20 = []
roll_std_list_20 = []

for i in range(len(x)):
    if i < win1:
        roll_mean_list_10.append(np.nan)
        roll_std_list_10.append(np.nan)
        roll_mean_list_20.append(np.nan)
        roll_std_list_20.append(np.nan)

    elif win1 <= i < win2:
        win_df = x['daily returns'].iloc[i-win1:i]
        mean = np.mean(win_df)
        std = np.std(win_df)

        roll_mean_list_10.append(mean)
        roll_std_list_10.append(std)

        roll_mean_list_20.append(np.nan)
        roll_std_list_20.append(np.nan)
    else:
        win1_df = x['daily returns'].iloc[i-win1:i]
        win2_df = x['daily returns'].iloc[i-win2:i]

        mean10 = np.mean(win1_df)
        std10 = np.std(win1_df)
        mean20 = np.mean(win2_df)
        std20 = np.std(win2_df)

        roll_mean_list_10.append(mean10)
        roll_mean_list_20.append(mean20)
        roll_std_list_10.append(std10)
        roll_std_list_20.append(std20)

x['10 day rolling mean'] = roll_mean_list_10
x['10 day rolling std'] = roll_std_list_10
x['20 day rolling mean'] = roll_mean_list_20
x['20 day rolling std'] = roll_std_list_20

In [44]:
pca = PCA(n_components=2)

features = [
    'daily returns',
    '10 day rolling mean',
    '10 day rolling std',
    '20 day rolling mean',
    '20 day rolling std'
]
x_cleaned = x[features].dropna()
result = pca.fit_transform(x_cleaned)

goog[['PC1', 'PC2']] = result

In [45]:
print("explained variance ratio:")
print(pca.explained_variance_ratio_)

print("PC1 explains", pca.explained_variance_ratio_[0])
print("PC2 explains", pca.explained_variance_ratio_[1])

print('total variance explained by pc1 and pc2')
print(pca.explained_variance_ratio_.sum())

explained variance ratio:
[0.60802624 0.28817366]
PC1 explains 0.6080262408675714
PC2 explains 0.28817365922659194
total variance explained by pc1 and pc2
0.8961999000941634


In [46]:
print(goog[['PC1', 'PC2']].head())

Price            PC1       PC2
Ticker                        
Date                          
2008-01-31  0.028394  0.015794
2008-02-01 -0.086691  0.015586
2008-02-04 -0.040731  0.025935
2008-02-05  0.021911  0.027439
2008-02-06 -0.011164  0.028759
